# Training Data

6 publicly-hosted NatureScot blogs with RSS feeds.

- [Scotland’s Nature](https://scotlandsnature.wordpress.com/)
- [Peatland ACTION Blog](https://peatlandaction.wordpress.com/blog/)
- [Scottish pollinators](https://scottishpollinators.wordpress.com/)
- [Forvie National Nature Reserve](https://forvienationalnaturereserve.home.blog/)
- [Isle of May National Nature Reserve](https://isleofmaynnr.wordpress.com/)
- [2 bogs, a swamp and some islands](https://2bogsaswampandsomeislands.wordpress.com/)

In [1]:
import requests
import feedparser
from bs4 import BeautifulSoup
import pickle
import urllib3

urllib3.disable_warnings()

## Download our training data

In [2]:
RSS_URLS = [
    'https://scotlandsnature.wordpress.com/feed/',
    'https://peatlandaction.wordpress.com/feed/',
    'https://scottishpollinators.wordpress.com/feed/',
    'https://forvienationalnaturereserve.home.blog/feed/',
    'https://isleofmaynnr.wordpress.com/feed/',
    'https://2bogsaswampandsomeislands.wordpress.com/feed/',
]

blogs = [requests.get(RSS_URL, verify=False) for RSS_URL in RSS_URLS]

## Clean our training data

- Find the text within the articles
- Perform some data cleaning
- Preprocess the text into sentences and words
- Save it for later

In [3]:
# Parse the XML RSS files
feeds = [feedparser.parse(x.text) for x in blogs]

# Find all the entries for each blog
feeds_entries = [feed.entries for feed in feeds]

# Add all the entries to one long list
entries = [entry for feed_entries in feeds_entries for entry in feed_entries]

# Grab the contents of each entry
entries_contents = [entry.content[0].value for entry in entries]

# Parse the HTML within the contents, like a web-browser
parsers = [
    BeautifulSoup(f'<body>{entry_contents}</body>', 'lxml-xml')
    for entry_contents in entries_contents
]

# Grab all the text from the contents.
texts = [parser.get_text() for parser in parsers]

# Perform some rudimentary data cleaning.
text = '. '.join(texts)
lines = text.split('\n')
lines_sentences = [line.split('. ') for line in lines]

# Retrieve a list of sentences from the data.
sentences = [sentence for line in lines_sentences for sentence in line if sentence != '']

# Split sentences into words.
sentences_words = [sentence.lower().split(' ') for sentence in sentences]

# Save it all for later
pickle.dump(sentences_words, open('sentences_words.pkl', 'wb'))